# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Memory Monitoring Utilities

In [3]:
def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def format_bytes(bytes):
    """Format bytes to human readable format"""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes < 1024.0:
            return f"{bytes:.2f} {unit}"
        bytes /= 1024.0
    return f"{bytes:.2f} PB"

def estimate_chunk_memory(width, height, bands, dtype):
    """Estimate memory requirement for a chunk"""
    dtype_sizes = {
        'uint8': 1, 'uint16': 2, 'uint32': 4,
        'int8': 1, 'int16': 2, 'int32': 4,
        'float32': 4, 'float64': 8
    }
    bytes_per_pixel = dtype_sizes.get(str(dtype), 4)
    return width * height * bands * bytes_per_pixel

def calculate_optimal_chunk_size(raster_width, raster_height, bands, dtype, memory_limit_mb=500):
    """Calculate optimal chunk size based on available memory"""
    memory_limit_bytes = memory_limit_mb * 1024 * 1024
    
    # Start with default chunk size
    chunk_size = 1024
    
    # Calculate memory for default chunk
    chunk_memory = estimate_chunk_memory(chunk_size, chunk_size, bands, dtype)
    
    # Adjust chunk size if needed
    if chunk_memory > memory_limit_bytes:
        # Calculate maximum chunk size that fits in memory
        bytes_per_pixel = chunk_memory / (chunk_size * chunk_size)
        max_pixels = memory_limit_bytes / bytes_per_pixel
        chunk_size = int(np.sqrt(max_pixels))
        # Round down to nearest power of 2 for efficiency
        chunk_size = 2 ** int(np.log2(chunk_size))
    
    # Ensure chunk size is at least 256
    chunk_size = max(256, chunk_size)
    
    print(f"📊 Optimal chunk size: {chunk_size}x{chunk_size}")
    print(f"   Estimated memory per chunk: {format_bytes(estimate_chunk_memory(chunk_size, chunk_size, bands, dtype))}")
    
    return chunk_size

print("✅ Memory monitoring utilities loaded")

✅ Memory monitoring utilities loaded


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [4]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [5]:

EVENT_NAME = '202405_Flood_Brasil'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'aria'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [51]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 8 .tif files in the S3 bucket.


['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [9]:
def makedirs(name):
    # Create necessary directories
    os.makedirs("reproj", exist_ok=True)
    
    # Create data_download directory for caching
    data_download_dir = "data_download"
    os.makedirs(data_download_dir, exist_ok=True)
    
    # Create subdirectory structure to match S3 path
    s3_path_parts = name.split('/')
    local_subdir = os.path.join(data_download_dir, *s3_path_parts[:-1])
    os.makedirs(local_subdir, exist_ok=True)

    # Local path for the downloaded file (persistent storage)
    local_download_path = os.path.join(data_download_dir, name)
    
    return data_download_dir, local_subdir, local_download_path

In [53]:
def convert_to_proper_CRS_and_cogify_chunked(name, cog_filename, cog_data_bucket, cog_data_prefix, 
                                            local_output_dir=None, chunk_config=None):
    """
    Convert a file to Cloud Optimized GeoTIFF with proper CRS using chunked processing.
    
    This function includes:
    - Chunked processing for memory efficiency
    - Download caching to avoid re-downloading files
    - CRS reprojection to EPSG:4326
    - COG validation before upload
    - Upload to S3
    - Smart nodata value handling based on data type
    - Memory monitoring and progress tracking
    """
    if chunk_config is None:
        chunk_config = CHUNK_CONFIG
    
    s3_key = f"{cog_data_prefix}/{cog_filename}"
    reproject_filename = f"reproj/{cog_filename}"

    #Make directories
    data_download_dir, local_subdir, local_download_path = makedirs(name)
    
    # Temporary file for processing
    temp_input_file = f"temp_{os.path.basename(name)}"
    
    # Memory monitoring
    if chunk_config.get('enable_memory_monitoring', True):
        initial_memory = get_memory_usage()
        print(f"   [MEMORY] Initial: {initial_memory:.1f} MB")

    try:
        import shutil
        
        # Check if file already exists locally
        if os.path.exists(local_download_path):
            print(f"   [CACHE HIT] Using cached file: {local_download_path}")
            shutil.copy(local_download_path, temp_input_file)
        else:
            # Download the file from S3
            print(f"   [DOWNLOAD] Downloading from S3...")
            s3_client.download_file(BUCKET, name, local_download_path)
            print(f"   [DOWNLOAD] ✅ Saved to cache")
            shutil.copy(local_download_path, temp_input_file)
        
        # Open source file and get metadata
        with rasterio.open(temp_input_file) as src:
            dst_crs = "EPSG:4326"
            chunk_size = chunk_config.get('default_chunk_size', 1024)
            
            # Check if reprojection is needed
            if src.crs and src.crs.to_string() == dst_crs:
                print(f"   [REPROJECT] Already in {dst_crs}, skipping reprojection")
                import shutil
                shutil.copy(temp_input_file, reproject_filename)
            else:
                print(f"   [REPROJECT] Converting to EPSG:4326 using chunked processing...")
                
                # Calculate transform for destination
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds
                )
                
                # Calculate optimal chunk size
                chunk_size = calculate_optimal_chunk_size(
                    width, height, src.count, src.dtypes[0],
                    memory_limit_mb=chunk_config.get('memory_limit_mb', 500)
                )
                
                # Prepare output kwargs
                kwargs = src.meta.copy()
                kwargs.update({
                    "driver": "GTiff",  # Use GTiff for intermediate file
                    "compress": "DEFLATE",
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height,
                    "tiled": True,
                    "blockxsize": min(chunk_size, width),
                    "blockysize": min(chunk_size, height)
                })
                
                # Create output file
                with rasterio.open(reproject_filename, "w", **kwargs) as dst:
                    # Calculate number of chunks
                    n_chunks_x = (width + chunk_size - 1) // chunk_size
                    n_chunks_y = (height + chunk_size - 1) // chunk_size
                    total_chunks = n_chunks_x * n_chunks_y
                    
                    print(f"   [CHUNKS] Processing {total_chunks} chunks ({n_chunks_x}x{n_chunks_y})")
                    
                    # Process each band
                    for band_idx in range(1, src.count + 1):
                        print(f"   [BAND {band_idx}/{src.count}] Processing...")
                        
                        # Use tqdm for progress tracking if enabled
                        if chunk_config.get('show_progress', True):
                            chunk_iterator = tqdm(
                                total=total_chunks,
                                desc=f"Band {band_idx}",
                                unit="chunks",
                                leave=False
                            )
                        else:
                            chunk_iterator = None
                        
                        # Process chunks
                        for y in range(0, height, chunk_size):
                            for x in range(0, width, chunk_size):
                                # Define window for this chunk
                                win_width = min(chunk_size, width - x)
                                win_height = min(chunk_size, height - y)
                                window = Window(x, y, win_width, win_height)
                                
                                # Create temporary arrays for chunk
                                chunk_data = np.zeros((win_height, win_width), dtype=src.dtypes[0])
                                
                                # Reproject chunk
                                reproject(
                                    source=rasterio.band(src, band_idx),
                                    destination=chunk_data,
                                    src_transform=src.transform,
                                    src_crs=src.crs,
                                    dst_transform=transform * rasterio.windows.transform(window, transform),
                                    dst_crs=dst_crs,
                                    resampling=Resampling.nearest,
                                    wrapdateline=True
                                )
                                
                                # Write chunk to output
                                dst.write(chunk_data, band_idx, window=window)
                                
                                # Update progress
                                if chunk_iterator:
                                    chunk_iterator.update(1)
                                
                                # Force garbage collection periodically
                                if (y // chunk_size * n_chunks_x + x // chunk_size) % 10 == 0:
                                    gc.collect()
                                    
                                    if chunk_config.get('enable_memory_monitoring', True):
                                        current_memory = get_memory_usage()
                                        if current_memory > initial_memory * 2:
                                            print(f"\n   [MEMORY] High usage: {current_memory:.1f} MB, forcing cleanup...")
                                            gc.collect()
                        
                        if chunk_iterator:
                            chunk_iterator.close()
        
        # COGify & upload
        print(f"   [COGIFY] Creating COG from reprojected file...")
        
        # Use rasterio to create COG
        with rasterio.open(reproject_filename) as src:
            # Smart nodata value handling based on data type
            print(f"   [NODATA] Data type: {src.dtypes[0]}")
            if src.dtypes[0] == 'uint8':
                nodata_value = 0
                print(f"   [NODATA] Using nodata value {nodata_value} for uint8 data")
            elif src.dtypes[0] == 'uint16':
                nodata_value = 0
                print(f"   [NODATA] Using nodata value {nodata_value} for uint16 data")
            else:
                nodata_value = -9999
                print(f"   [NODATA] Using nodata value {nodata_value} for {src.dtypes[0]} data")
            
            # Update profile for COG
            profile = src.profile.copy()
            profile.update(COG_PROFILE)
            profile['nodata'] = nodata_value
            
            with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
                tmp_name = tmp.name
                
                # Write COG using chunked approach
                with rasterio.open(tmp_name, 'w', **profile) as dst:
                    # Process in chunks to avoid memory issues
                    for band_idx in range(1, src.count + 1):
                        for y in range(0, src.height, chunk_size):
                            for x in range(0, src.width, chunk_size):
                                win_width = min(chunk_size, src.width - x)
                                win_height = min(chunk_size, src.height - y)
                                window = Window(x, y, win_width, win_height)
                                
                                data = src.read(band_idx, window=window)
                                dst.write(data, band_idx, window=window)
                
                # Validate COG
                print(f"   [VALIDATE] Checking COG validity...")
                is_valid_cog, validation_details = validate_cog(tmp_name)
                
                if is_valid_cog:
                    print(f"   [VALIDATE] ✅ Valid COG")
                else:
                    print(f"   [VALIDATE] ⚠️ COG validation warnings")
                    critical_errors = [e for e in validation_details.get('errors', []) if 'Invalid driver' in e]
                    if critical_errors:
                        raise ValueError(f"Critical COG validation failed")
                    if 'errors' in validation_details:
                        for error in validation_details['errors']:
                            print(f"      - {error}")
                    if 'warnings' in validation_details:
                        for warning in validation_details['warnings']:
                            print(f"      - {warning}")
                
                # Upload to S3
                print(f"   [UPLOAD] Uploading to S3...")
                s3_client.upload_file(
                    Filename=tmp_name,
                    Bucket=cog_data_bucket,
                    Key=s3_key
                )
                print(f"   [SUCCESS] ✅ Uploaded to s3://{cog_data_bucket}/{s3_key}")
                
                # Save locally if specified
                if local_output_dir:
                    os.makedirs(local_output_dir, exist_ok=True)
                    local_path = os.path.join(local_output_dir, cog_filename)
                    import shutil
                    shutil.copy(tmp_name, local_path)
        
        # Final memory report
        if chunk_config.get('enable_memory_monitoring', True):
            final_memory = get_memory_usage()
            print(f"   [MEMORY] Final: {final_memory:.1f} MB (Change: {final_memory - initial_memory:+.1f} MB)")
            
    except Exception as e:
        print(f"   [ERROR] Failed: {str(e)}")
        raise
            
    finally:
        # Clean up temporary files
        for temp_file in [temp_input_file, reproject_filename]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        if 'tmp_name' in locals() and os.path.exists(tmp_name):
            os.remove(tmp_name)
        
        # Force final garbage collection
        gc.collect()

print("✅ Chunked COG conversion function defined with memory-efficient processing")

✅ Chunked COG conversion function defined with memory-efficient processing


In [11]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 248
  - Total size: 7.37 GB

📁 Cached files (first 10):
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002653_DVR_RTC20_G_gpuned_0610_WM.tif (3.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002653_DVR_RTC20_G_gpuned_0610_rgb.tif (258.2 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002719_DVR_RTC20_G_gpuned_F141_WM.tif (2.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002719_DVR_RTC20_G_gpuned_F141_rgb.tif (289.2 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240507T122323_DVR_RTC20_G_gpuned_5BA0_WM.tif (2.7 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240507T122323_DVR_RTC20_G_gpuned_5BA0_rgb.tif (321.6 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240512T002655_DVR_RTC20_G_gpuned_EC9C_WM.tif (9.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240512T002720_DVR_RTC20_G_gpuned_D32B_WM.tif (

(248, 7916379572)

In [33]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    
    print_batch_summary(results)
    return results

# Process files

In [13]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [17]:
# Define filename creator functions for different file types

def create_cog_filename_floodmap(f, EVENT_NAME):
    """Create COG filename for FloodMap files, moving FloodMap before dates."""
    f2 = Path(f).stem
    
    # Check if it has FloodMap at the end
    if '_FloodMap' in f2:
        # Remove _FloodMap from the end
        f2_no_floodmap = f2.replace('_FloodMap', '')
        
        # Split by underscore
        parts = f2_no_floodmap.split('_')
        
        # Find the date range part (YYYYMMDD-YYYYMMDD)
        date_index = None
        for i, part in enumerate(parts):
            if '-' in part and len(part) == 17 and part[:8].isdigit() and part[9:].isdigit():
                date_index = i
                date_range = part
                # Format dates
                date1, date2 = date_range.split('-')
                formatted_dates = f"{date1[:4]}-{date1[4:6]}-{date1[6:8]}_{date2[:4]}-{date2[4:6]}-{date2[6:8]}"
                break
        
        if date_index is not None:
            # Reconstruct with FloodMap before dates
            base_parts = parts[:date_index]
            new_name = '_'.join(base_parts) + '_FloodMap_' + formatted_dates
            cog_filename = f'{EVENT_NAME}_{new_name}day.tif'
        else:
            # Fallback if format is unexpected
            cog_filename = f'{EVENT_NAME}_{f2}day.tif'
    else:
        # Fallback if no FloodMap found
        cog_filename = f'{EVENT_NAME}_{f2}day.tif'
    
    return cog_filename

filter_str = 'FloodMap'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_floodmap(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-05-06_2024-04-21day.tif


In [18]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_floodmap, 
                                target_dir = "HLS/aria", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-05-06_2024-04-21day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/aria

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/1] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif
   Output filename: 202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-05-06_2024-04-21day.tif
   [MEMORY] Initial: 298.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [CHUNKS] Processing 104 chunks (13x8)
   [BAND 1/1] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-05-06_2024-04-21day.tif
   [MEMORY] Final: 460.5 MB (Change: +161.7 MB)
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-05-06_2024-04-21day.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/aria/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/aria/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-04T1

In [19]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [34]:
# Define filename creator functions for different file types

def create_cog_filename_dswx_hls_mosaic(f, EVENT_NAME):
    """Create COG filename for DSWx-HLS mosaic files, moving suffix before date."""
    f2 = Path(f).stem
    
    # Split by hyphen to separate the main parts
    parts = f2.split('-')
    
    # Find the date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Get everything before the date
        prefix_parts = parts[:date_index]
        # Get everything after the date
        suffix_parts = parts[date_index + 1:]
        
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Reconstruct: prefix + suffix + date
        prefix = '-'.join(prefix_parts)
        suffix = '-'.join(suffix_parts) if suffix_parts else ""
        
        if suffix:
            # Replace hyphens with underscores for consistency
            prefix = prefix.replace('-', '_')
            suffix = suffix.replace('-', '_')
            cog_filename = f'{EVENT_NAME}_{prefix}_{suffix}_{formatted_date}day.tif'
        else:
            prefix = prefix.replace('-', '_')
            cog_filename = f'{EVENT_NAME}_{prefix}_{formatted_date}day.tif'
    else:
        # Fallback if format is unexpected
        f2_cleaned = f2.replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{f2_cleaned}day.tif'
    
    return cog_filename


filter_str = 'S2B_L8'  # This will catch files starting with dates in the DSWx folder

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_dswx_hls_mosaic(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06day.tif


In [23]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_dswx_hls_mosaic, 
                                target_dir = "HLS/opera", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/opera

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/1] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif
   Output filename: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06day.tif
   [MEMORY] Initial: 461.2 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [CHUNKS] Processing 440 chunks (22x20)
   [BAND 1/1] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/opera/202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06day.tif
   [MEMORY] Final: 799.2 MB (Change: +338.0 MB)
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06day.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/opera/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/opera/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-04T15:0

In [24]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [37]:
# Define filename creator functions for different file types
import re 
def create_cog_filename_dswx_hls_timestamp(f, EVENT_NAME):
    """Create COG filename for DSWx-HLS files with timestamp, moving date to end."""
    f2 = Path(f).stem
    
    # Split by underscore
    parts = f2.split('_')
    
    # Find the part with timestamp (YYYYMMDDTHHMMSSZ format)
    timestamp_index = None
    timestamp_str = None
    
    for i, part in enumerate(parts):
        if 'T' in part and 'Z' in part and len(part) >= 16:
            # Extract just the date part from timestamp
            timestamp_index = i
            timestamp_str = part
            break
    
    if timestamp_index is not None and timestamp_str:
        # Extract date from timestamp (YYYYMMDD from YYYYMMDDTHHMMSSZ)
        date_part = timestamp_str[:8]
        time_part = timestamp_str[9:15]  # HHMMSS
        
        # Format date and time
        formatted_datetime = f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]}T{time_part[:2]}:{time_part[2:4]}:{time_part[4:6]}Z"
        
        # Remove timestamp from parts
        parts.pop(timestamp_index)
        
        # Reconstruct filename with date at the end
        base_name = '_'.join(parts).replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{base_name}_{formatted_datetime}.tif'
    else:
        # Fallback if format is unexpected
        f2_cleaned = f2.replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{f2_cleaned}day.tif'
    
    return cog_filename

pattern = re.compile(r'(?=.*S2A)(?=.*mosaic)')
filter_ = [f for f in keys if re.search(pattern, f)]

# Test functions
print("Testing WM filename:")

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_dswx_hls_timestamp(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_mosaic_2024-04-21T13:31:51Z.tif
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-18T13:22:31Z.tif
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif


In [38]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_dswx_hls_timestamp, 
                                target_dir = "HLS/opera", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_mosaic_2024-04-21T13:31:51Z.tif
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-18T13:22:31Z.tif
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/opera

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/3] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif
   Output filename: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_mosaic_2024-04-21T13:31:51Z.tif
   [MEMORY] Initial: 802.1 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [CHUNKS] Pro

   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/opera/202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_mosaic_2024-04-21T13:31:51Z.tif
   [MEMORY] Final: 804.0 MB (Change: +1.9 MB)
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_mosaic_2024-04-21T13:31:51Z.tif

[2/3] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif
   Output filename: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-18T13:22:31Z.tif
   [MEMORY] Initial: 804.0 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk si

   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/opera/202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-18T13:22:31Z.tif
   [MEMORY] Final: 590.2 MB (Change: -213.8 MB)
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-18T13:22:31Z.tif

[3/3] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif
   Output filename: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif
   [MEMORY] Initial: 590.2 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal 

   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/opera/202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif
   [MEMORY] Final: 610.2 MB (Change: +20.0 MB)
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif

✅ Batch processing complete: 3 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/opera/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/opera/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 3
Successful: 3
Failed: 0
Success rate: 100.0%
Timestamp: 20

In [39]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [41]:

def create_cog_filename_dswx_hls_archive(f, EVENT_NAME):
    """Create COG filename for archive DSWx-HLS files, moving date to the end."""
    f2 = Path(f).stem
    
    # Split by underscore
    parts = f2.split('_')
    
    # Find the date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Remove date from parts
        parts.pop(date_index)
        
        # Reconstruct with date at the end
        base_name = '_'.join(parts).replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{base_name}_{formatted_date}day.tif'
    else:
        # Fallback if format is unexpected
        f2_cleaned = f2.replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{f2_cleaned}day.tif'
    
    return cog_filename


filter_str = 'archive'
# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_dswx_hls_archive(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-04-21day.tif
  202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06day.tif


In [42]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_dswx_hls_archive, 
                                target_dir = "HLS/archive", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-04-21day.tif
  202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/archive

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/2] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif
   Output filename: 202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-04-21day.tif
   [MEMORY] Initial: 611.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [CHUNKS] Processing 252 chunks (18x14)
   [BAND 1/1] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/archive/202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-04-21day.tif
   [MEMORY] Final: 652.4 MB (Change: +40.4 MB)
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-04-21day.tif

[2/2] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif
   Output filename: 202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06day.tif
   [MEMORY] Initial: 652.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chun

   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/archive/202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06day.tif
   [MEMORY] Final: 694.5 MB (Change: +42.2 MB)
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06day.tif

✅ Batch processing complete: 2 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/archive/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/archive/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 20

In [43]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [54]:
def create_cog_filename_flood_depth(f, EVENT_NAME):
    """Create COG filename for flood depth files, adding event date from EVENT_NAME."""
    f2 = Path(f).stem
    
    # Extract date from EVENT_NAME (format: YYYYMM_EventType_Location)
    event_parts = EVENT_NAME.split('_')
    if event_parts and len(event_parts[0]) == 6 and event_parts[0].isdigit():
        event_date = event_parts[0]  # YYYYMM
        # Format as YYYY-MM
        formatted_date = f"{event_date[:4]}{event_date[4:6]}"
        
        # Create filename with event date at the end
        cog_filename = f'{EVENT_NAME}_{f2}_{formatted_date}month.tif'
    else:
        # Fallback if EVENT_NAME doesn't have expected format
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename


filter_str = 'flood_depth'
# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_flood_depth(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202405_Flood_Brasil_FwDET_GEE_FwDET_Brazil_202405month.tif


In [55]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_flood_depth, 
                                target_dir = "GoogleEarth", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_FwDET_GEE_FwDET_Brazil_202405month.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/GoogleEarth

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/1] Processing: drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif
   Output filename: 202405_Flood_Brasil_FwDET_GEE_FwDET_Brazil_202405month.tif
   [MEMORY] Initial: 694.6 MB
   [CACHE HIT] Using cached file: data_download/drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'def

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")